# Stage 2 — end-to-end smoke run

Personalizing `ivrit-ai/whisper-large-v3` — the arm-B model from Stage 1 — to a single
speaker. This notebook runs
**every stage of the Stage 2 pipeline once**, on a deliberately small slice, to answer
one question: *does the code work end to end?* It is not a research run — the budgets
are tiny and the results mean nothing.

The design this implements is in [`stage2/plan.html`](plan.html); the code is
`pipeline.py`, `train.py`, `evaluate.py`.

**Order.** `chunk` is text-only and `files_needed` depends on the split, so the practical
order differs from the plan's stage numbering:

    index → chunk → split → materialize → baseline → train → eval → stats
     01       03      04        02           05        06      07      08

**Runtime.** ~30–50 min on a free T4, most of it audio download and the base-model
transcription. Set `Runtime → Change runtime type → GPU` before starting.

## 0 · Setup

In [ ]:
# --- GPU check. Nothing below trains without one. ---
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(out or 'NO GPU  ->  Runtime > Change runtime type > T4 GPU, then rerun')

In [ ]:
# --- Dependencies. transformers 5 is required: the code uses `dtype=`,
# `eval_strategy`, and assumes forced_decoder_ids is gone (it was removed in v5). ---
%pip -q install -U "transformers>=5.0,<6" "peft>=0.17" "huggingface_hub>=0.30" \
                   rapidfuzz pyarrow scipy

import transformers, peft, torch
print('transformers', transformers.__version__, '| peft', peft.__version__,
      '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert int(transformers.__version__.split('.')[0]) >= 5, \
    'transformers < 5 -- Runtime > Restart session, then run this cell again'

If the assert fired, do **Runtime → Restart session** and rerun the cell. Colab pins an
older `transformers` at boot and the upgrade only takes effect after a restart.

In [ ]:
# --- Repo ---
import os, sys
REPO_DIR = '/content/deep-learning-project'
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/hadasy-tau/deep-learning-project.git {REPO_DIR}
%cd {REPO_DIR}/stage2
sys.path.insert(0, os.getcwd())
!git -C {REPO_DIR} log --oneline -1

In [ ]:
# --- Hugging Face auth. VoxKnesset is gated: needed for WAVEFORMS ONLY (the
# `index` and `chunk` stages read a public dump and need no token).
# Put your token in Colab's secrets panel (key icon) as HF_TOKEN. ---
from huggingface_hub import login, whoami
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    tok = getpass('HF token: ')
login(token=tok, add_to_git_credential=False)
print('logged in as:', whoami()['name'])

In [ ]:
# --- OPTIONAL but recommended: persist audio + the shard index across sessions.
# Without this you re-download several GB every time the runtime dies. ---
USE_DRIVE = False   # set True to mount

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/voxknesset_stage2'
else:
    CACHE = '/content/cache'
os.makedirs(CACHE, exist_ok=True)
AUDIO_DIR = os.path.join(CACHE, 'audio'); os.makedirs(AUDIO_DIR, exist_ok=True)
print('cache:', CACHE)

## 1 · Configuration

Every knob for the run lives here. The defaults are chosen so the whole notebook
finishes on a free T4.

**Model.** `ivrit-ai/whisper-large-v3` — the same model Stage 1 measured as arm B, and the
one this project exists to personalize. That continuity matters: every arm-B number you
have (`wer_B`, the speaker error map, `stage0_gate`'s clustering estimates) came from these
weights, and `like_for_like()` later asserts the base model reproduces that speaker's
Stage-1 `wer_B`. Swap the backbone and that gate fails for reasons unrelated to any bug.

Stage 1 ran inference through the CTranslate2 conversion (`-ct2`) for speed; these are the
same weights in a different runtime, and `-ct2` cannot be fine-tuned, so training uses the
transformers checkpoint.

1.55 B on a 16 GB T4 is tight but works with LoRA + fp16 + gradient checkpointing at
batch 1 — which is what the plan's compute table calls "fine for verifying the pipeline
runs". If you still hit OOM, `ivrit-ai/whisper-large-v3-turbo` (809 M) is the fallback,
with the caveat that its decoder is far shallower so **site-axis conclusions do not
transfer** and the baseline stops matching Stage 1.

**Speaker.** `11835` is S1 from the panel — high `wer_B`, low `gain_rel`, the speaker the
population fine-tune served worst. 27 h across 311 sessions, so there is plenty to slice.

In [ ]:
# --- Run configuration ---
SPEAKER   = 11835          # S1: high wer_B, low gain_rel
MODEL_B   = 'ivrit-ai/whisper-large-v3'    # arm B: what Stage 1 measured, what we personalize
# OOM fallback only: 'ivrit-ai/whisper-large-v3-turbo' (809M, breaks Stage-1 comparability)

N_SESSIONS = 14            # newest sessions to pull for this speaker (caps the download)
TEST_MIN   = 8             # personal-test minutes  (real runs: sized by stage0_gate)
DEV_MIN    = 4             # personal-dev minutes
BUDGET_MIN = 3             # training budget in minutes of speech
PULL_TRAIN_MIN = 12        # train minutes to DOWNLOAD (> BUDGET_MIN, leaves headroom)

SITE, METHOD, RANK, LR = 'both', 'lora', 8, 1e-3
EPOCHS, BATCH, GRAD_ACCUM = 3, 1, 8            # batch 1 + accumulation: T4 has 16 GB
EVAL_CHUNKS = 40           # cap test chunks scored, to keep inference short

# Point both modules at the chosen model. ARMS is a dict in each module, so this
# is the supported way to swap the backbone without editing the source.
import train, evaluate as E
train.ARMS['B'] = E.ARMS['B'] = MODEL_B
print('arm B ->', MODEL_B)

## 2 · `index` — what the corpus contains

The public ct2 dump carries `reference_text`, `segments_json`, durations and every
demographic column, so this stage needs no token and touches none of VoxKnesset's 275 GB.

`local_only=False` matters here: the default is `True`, which assumes a warm HF cache
that a fresh Colab runtime does not have.

In [ ]:
import pipeline as P, pandas as pd, numpy as np
pd.set_option('display.width', 200)

idx = P.load_index(local_only=False)          # ~1-2 min on first run
print(f'{len(idx):,} recordings, {idx.speaker_id.nunique()} speakers')
idx[['filename', 'speaker_id', 'session', 'age', 'duration_s']].head()

In [ ]:
# This speaker's footprint, and the slice we will actually pull.
spk = idx[idx.speaker_id == SPEAKER].sort_values('session')
print(f'speaker {SPEAKER}: {len(spk):,} recordings, {spk.session.nunique()} sessions, '
      f'{spk.duration_s.sum()/3600:.1f} h total')

sessions = sorted(spk.session.unique())[-N_SESSIONS:]     # newest N sessions
sub = spk[spk.session.isin(sessions)]
print(f'slice: {len(sub)} recordings across {len(sessions)} sessions, '
      f'{sub.duration_s.sum()/60:.0f} min')

## 3 · `chunk` — protocol text aligned to audio

Whisper's window is 30 s; VoxKnesset segments average ~124 s and carry no timestamps.
This stage cuts the reference text into ≤ 30 s pieces with approximate times, using the
hypothesis segmentation in `segments_json` as an anchor. Text only — no audio needed yet.

Each chunk gets an **alignment score**, which is stored and deliberately **never filtered
on**: filtering it would delete exactly the hard cases and let arm B curate the test set
both arms are judged on. The `wpm` filter (30–350) stays, because it is model-independent.

In [ ]:
ch = P.chunk_all(sub)
print(f'{len(ch):,} chunks from {len(sub)} recordings')
print(f'duration  : median {ch.duration_s.median():.1f}s  max {ch.duration_s.max():.1f}s')
print(f'alignment : median {ch.score.median():.3f}  p10 {ch.score.quantile(.1):.3f}')
assert (ch.duration_s <= 30 + 1e-6).all(), 'chunk longer than the Whisper window'
ch[['filename', 'session', 'start', 'end', 'duration_s', 'n_words', 'score']].head()

In [ ]:
# Alignment quality is worth eyeballing, not acting on.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
ax[0].hist(ch.score, bins=40, color='#0072B2'); ax[0].set_title('alignment score')
ax[1].hist(ch.duration_s, bins=40, color='#D55E00'); ax[1].set_title('chunk duration (s)')
for a in ax: a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 4 · `split` — session-disjoint

Sessions are ordered by `age` and cut chronologically: newest → personal-test, then dev,
the rest train. **Never split randomly within a session** — same room, same mic, same
topic, adjacent audio; a random split measures session memorization, not personalization.

Real runs size personal-test per speaker from `stage0_gate` (40 min at WER .19, ~7 h at
WER .06). `TEST_MIN` here is far smaller and the resulting numbers resolve nothing — that
is fine for a smoke test and not fine for anything else.

In [ ]:
sp = P.make_splits(ch, test_min=TEST_MIN, dev_min=DEV_MIN)
print(sp.groupby('part').agg(chunks=('filename', 'size'),
                             minutes=('duration_s', lambda s: round(s.sum()/60, 1)),
                             sessions=('session', 'nunique')).to_string())

# the guarantee the whole design rests on
by = {p: set(g.session) for p, g in sp.groupby('part')}
for a, b in (('train', 'test'), ('train', 'dev'), ('dev', 'test')):
    assert not (by.get(a, set()) & by.get(b, set())), f'{a}/{b} session overlap'
print('\nsession-disjoint: OK')

In [ ]:
# The training budget is a prefix of one ordering, so nested budgets are nested by
# construction rather than by a separate carving step.
tr = train.take_budget(sp[sp.part == 'train'], BUDGET_MIN)   # sp here: pre-download
print(f'budget {BUDGET_MIN} min -> {len(tr)} chunks, {tr.duration_s.sum()/60:.1f} min, '
      f'{tr.session.nunique()} sessions')

## 5 · `materialize` — the only stage that needs the gated corpus

Two things make this affordable. Audio is fetched **by split** (dev + test + the training
budget) rather than by speaker — the design only ever trains on minutes, so pulling a
speaker's whole 27 h would be waste. And `audio.path` equals `filename`, which lets a
cheap index locate each recording's shard and row group without reading a waveform.

Parquet's smallest readable unit is a row group, so the transfer is somewhat larger than
the audio actually kept.

**First run builds the shard index (~18 min).** With `USE_DRIVE = True` it is cached and
this cost is paid once, ever.

In [ ]:
import shutil, os
# reuse a cached shard index if one is in Drive
cached_idx = os.path.join(CACHE, 'shard_index.csv')
if os.path.exists(cached_idx) and not os.path.exists('shard_index.csv'):
    shutil.copy(cached_idx, 'shard_index.csv'); print('shard index restored from cache')

need = P.files_needed(sp, train_min=PULL_TRAIN_MIN)
mins = idx.set_index('filename').duration_s.reindex(need).sum() / 60
print(f'{len(need)} recordings needed, {mins:.0f} min of audio')

In [ ]:
n = P.materialize(need, AUDIO_DIR)       # first call may spend ~18 min building the index
print(f'\n{n} files written to {AUDIO_DIR}')

if not os.path.exists(cached_idx) and os.path.exists('shard_index.csv'):
    shutil.copy('shard_index.csv', cached_idx)     # keep it for next session
print('on disk:', len(os.listdir(AUDIO_DIR)), 'wav files',
      f"({sum(os.path.getsize(os.path.join(AUDIO_DIR, f)) for f in os.listdir(AUDIO_DIR))/1e9:.2f} GB)")

In [ ]:
# Chunk times come from the dump's duration_s, so confirm they agree with the real
# waveform -- a mismatch would slice past end of file.
import wave
have = sp[sp.filename.isin(os.listdir(AUDIO_DIR))]
for r in have.drop_duplicates('filename').head(20).itertuples():
    with wave.open(os.path.join(AUDIO_DIR, r.filename), 'rb') as w:
        real = w.getnframes() / w.getframerate()
    assert real > 0
# Everything downstream uses sp_ok, not sp: chunk rows whose recording is on disk.
# overfit_check takes .head(n) of the train side in frame order, which is NOT the
# budget order materialize used, so passing the full frame would read missing files.
sp_ok = sp[sp.filename.isin(set(os.listdir(AUDIO_DIR)))].reset_index(drop=True)
print(f'{have.filename.nunique()} materialized files, {len(sp_ok)} chunks backed by audio')
print(sp_ok.groupby('part').agg(chunks=('filename','size'),
                                minutes=('duration_s', lambda s: round(s.sum()/60,1))).to_string())
assert (sp_ok.part == 'train').any() and (sp_ok.part == 'test').any(), 'need train and test audio'

## 6 · Training sanity — overfit 20 examples

Before any real run: a correct setup drives a 20-example subset to near-zero loss. If this
does not fall, nothing downstream is worth running and the problem is in the training code,
not the data.

In [ ]:
losses = train.overfit_check(sp_ok, AUDIO_DIR, speaker=SPEAKER, arm='B', n=20, steps=60)

## 7 · `baseline` — score personal-test before touching the model

Short-form (per chunk) is the primary protocol: it matches the training distribution and
yields far more test items for the paired bootstrap. Long-form on whole recordings is the
secondary protocol and the only figure comparable to Stage 1.

Error **counts** are stored, never rates, so speaker WER stays `total_errors / total_words`
however the rows are later grouped.

In [ ]:
test = sp_ok[sp_ok.part == 'test'].head(EVAL_CHUNKS).reset_index(drop=True)
print(f'scoring {len(test)} test chunks, {test.duration_s.sum()/60:.1f} min')

model, proc, dev = E.load(arm='B')
hyp_base = E.transcribe_short(model, proc, test, AUDIO_DIR, batch=8, device=dev)
base = E.score(test.text.tolist(), hyp_base)
print(f'\nbase WER {E.wer(base):.4f}   ({base.werr.sum()} errors / {base.n_words.sum()} words)')
base.head()

In [ ]:
# Eyeball a few. Hebrew is RTL in the output; mismatches are usually obvious anyway.
for i in range(3):
    print(f'REF: {test.text[i][:110]}')
    print(f'HYP: {hyp_base[i][:110]}')
    print(f'     werr={base.werr[i]}  S={base.S[i]} D={base.D[i]} I={base.I[i]}\n')

## 8 · `train` — one cell

A cell is `(speaker, arm, site, method, budget, rank, lr, seed)` — the axes of D4. This runs
exactly one, at the reference config. `train_cell` is idempotent: it writes a `DONE` marker
and skips a completed cell, so re-running this notebook does not retrain.

In [ ]:
free = torch.cuda.mem_get_info()[0] / 1e9 if torch.cuda.is_available() else 0
print(f'{free:.1f} GB free before training')

out = train.train_cell(sp_ok, AUDIO_DIR, out_root=os.path.join(CACHE, 'runs'),
                       speaker=SPEAKER, arm='B', site=SITE, method=METHOD,
                       budget=BUDGET_MIN, rank=RANK, lr=LR, seed=0,
                       epochs=EPOCHS, batch=BATCH, grad_accum=GRAD_ACCUM)
print('\nadapter ->', out)

## 9 · `eval` — the same segments, the tuned model

The comparison is paired: base and tuned scored on **the same chunks**. That is what makes
the bootstrap in the next cell meaningful, and it is why `transcribe_short` is called again
here rather than reusing anything.

In [ ]:
import gc
for v in ('model', 'tuned_model'):
    if v in dir(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()
print(f'{torch.cuda.mem_get_info()[0]/1e9:.1f} GB free')

tuned_model, tuned_proc, dev = E.load(arm='B', adapter=out)
hyp_tuned = E.transcribe_short(tuned_model, tuned_proc, test, AUDIO_DIR, batch=8, device=dev)
tuned = E.score(test.text.tolist(), hyp_tuned)
print(f'base  WER {E.wer(base):.4f}')
print(f'tuned WER {E.wer(tuned):.4f}')

## 10 · `stats` — paired bootstrap

`stage0_gate` said in advance what a test set this size can resolve. With `TEST_MIN` at
smoke-test scale the answer is *almost nothing*, so expect a wide interval straddling zero.
**That is the correct outcome here** and says nothing about whether personalization works —
it says the test set is tiny, which we chose on purpose.

The S/D/I breakdown is not decoration: if gains come mostly from fewer **insertions**, the
model learned protocol style rather than the speaker.

In [ ]:
r = E.paired_bootstrap(base, tuned, n_boot=2000)
for k, v in r.items():
    print(f'  {k:14s} {v:.4f}' if isinstance(v, float) else f'  {k:14s} {v}')

print('\nerror composition:')
comp = pd.DataFrame({'base': [base.S.sum(), base.D.sum(), base.I.sum()],
                     'tuned': [tuned.S.sum(), tuned.D.sum(), tuned.I.sum()]},
                    index=['substitutions', 'deletions', 'insertions'])
comp['delta'] = comp.tuned - comp.base
print(comp.to_string())
print(f'\nrunaway decodes: base {base.runaway.sum()}  tuned {tuned.runaway.sum()}')

## What this run does and does not tell you

**Does:** every stage executes, the split is session-disjoint, chunks are ≤ 30 s and backed
by real audio, the training loop reduces loss, an adapter loads and changes the output, and
the statistics run on paired counts.

**Does not:** anything about whether personalization helps. The budget is minutes, the test
set is minutes, and the backbone is turbo rather than large-v3.

### Before this becomes a research run

1. **Size personal-test from `stage0_gate`** — `stage0/outputs/stage0_gate_curve.csv` has
   the minimum detectable effect per test size, per speaker. For a 10% relative target:
   S1 40 min, S2 1.8 h, S3 6.1 h, S4 7.2 h.
2. **Run the like-for-like gate** — `E.like_for_like()` re-scores personal-test in
   long-form with the base model and must reproduce that speaker's Stage-1 `wer_B`. If it
   fails, chunking or normalization changed the metric and every comparison is invalid.
3. **Add the cross-speaker control** — "train on S, WER drops" conflates speaker acoustics
   with protocol style, session memorization and topic drift. Δ personal − Δ domain is the
   personalization effect; Δ personal alone is not.
4. **A T4 cannot full fine-tune 1.55 B.** The site axis needs a 40 GB card or it collapses
   into a comparison of LoRA variants.